# 03. 시황: 밸류체인 전체를 한눈에

**목표:** 01·02가 한 기업을 깊게 본다면, 여기서는 **같은 밸류체인에 있는 기업들의 주가·밸류에이션·뉴스**를 넓게 본다. "시장은 지금 무엇을 믿고 있나"를 숫자로 확인하는 단계다.

- 종목 목록은 플레이북의 `universe`에서 온다 ([playbooks/neocloud.yaml](../playbooks/neocloud.yaml)). 역할별 분류는 스크리닝용 작업 가정이다.
- 수익률은 **현지 통화 기준**, 각 종목의 자국 지수와 비교한다 (일본 Nikkei 225, 한국 KOSPI, 대만 TAIEX, 홍콩 Hang Seng).
- 데이터는 Yahoo Finance. 그날 받은 값을 `data/cache/market/<날짜>/`에 저장해서 같은 날 다시 실행하면 같은 숫자가 나온다.
- **주의:** Yahoo 밸류에이션 값은 기준이 제각각이고 가끔 틀린다. 투자 판단의 입력이 아니라 '어디를 더 볼지' 고르는 용도다.

In [ ]:
import sys, json, re
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from research import financials as fin, playbook

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"font.family": ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"],
                     "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"   # 범주 색 고정 순서
from research import market as mk, playbook

PLAYBOOK = "neocloud"
REFRESH = False          # True: 오늘 데이터를 새로 받기
pb = playbook.load(PLAYBOOK)
groups = pb["universe"]
tickers = [t for g in groups.values() for t in g]
names = {t: n for g in groups.values() for t, n in g.items()}
role = {t: r for r, g in groups.items() for t in g}
px = mk.prices(tickers, refresh=REFRESH)
print(f"{len(tickers)}개 종목, 기준일 {px.index[-1].date()}")

## 1. 주가 성과

`1년 지수 대비(%p)`가 음수면 자국 시장보다 못했다는 뜻이다. `52주 고점 대비`는 고점에서 얼마나 내려와 있는지, `연율 변동성`은 주가가 얼마나 크게 흔들리는지를 보여 준다.

In [ ]:
perf = mk.performance(px, tickers)
perf.insert(0, "역할", [role[t] for t in perf.index])
perf.insert(0, "기업", [names[t] for t in perf.index])
num = [c for c in perf.columns if c not in ("기업", "역할", "지수", "기준일")]
perf[num] = perf[num].astype(float)
perf.style.format({c: "{:+.1f}" for c in num}, na_rep="-").background_gradient(
    subset=["1년 지수 대비(%p)"], cmap="RdBu", vmin=-60, vmax=60)

## 2. 역할별 주가 흐름 (1년 전 = 100)

역할마다 차트 하나, 축 하나. 회색 점선은 해당 그룹 첫 종목의 자국 지수다.

In [ ]:
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
rb = mk.rebased(px, list(px.columns), months=12)
n = len(groups); cols = 2; rows = -(-n // cols)
fig, axes = plt.subplots(rows, cols, figsize=(13, 3.8 * rows), squeeze=False)
for a, (g, members) in zip(axes.flat, groups.items()):
    for c, t in zip(COLORS, members):
        a.plot(rb.index, rb[t], color=c, lw=2, label=names[t])
    bm = mk.benchmark_for(list(members)[0])
    if bm and bm[0] in rb:
        a.plot(rb.index, rb[bm[0]], color="#8a8985", lw=1.5, ls="--", label=bm[1])
    a.axhline(100, color="#52514e", lw=0.8)
    a.set_title(g, loc="left"); a.legend(frameon=False, fontsize=9, loc="upper left")
for a in list(axes.flat)[n:]:
    a.set_visible(False)
fig.tight_layout(); plt.show()

## 3. 밸류에이션 스냅샷

시가총액은 현지 통화. 적자 기업은 PER이 비어 있거나 음수다. 🔎 표시는 값이 비정상적으로 커서(PER 100배 초과 또는 음수) **원 자료 확인이 필요한 칸**이다.

In [ ]:
val = mk.valuation(tickers, refresh=REFRESH)
val.insert(0, "역할", [role[t] for t in val.index])
val.insert(0, "기업", [names[t] for t in val.index])
val["시가총액(억)"] = val.pop("시가총액") / 1e8
odd = lambda v: "🔎" if pd.notna(v) and (v > 100 or v < 0) else ""
for c in ("PER(지난 12개월)", "PER(예상)"):
    val[c + " 확인"] = val[c].map(odd)
val.style.format({c: "{:,.1f}" for c in val.columns if val[c].dtype.kind == "f"}, na_rep="-")

### 역할별 중앙값

같은 역할 안에서 비교해야 의미가 있다. 서버 제조사와 클라우드 운영사의 PER을 바로 비교하면 안 된다.

In [ ]:
cols_ = ["PER(지난 12개월)", "PER(예상)", "PBR", "EV/EBITDA", "EV/매출"]
clean = val[cols_].where((val[cols_] > 0) & (val[cols_] < 100))  # 🔎 값은 빼고
clean.groupby(val["역할"]).median().style.format("{:.1f}", na_rep="-")

## 4. 최근 헤드라인

Yahoo가 종목에 연결한 기사다. **종목과 직접 관련 없는 기사도 섞여 있다** (예: 업계 일반 기사). 헤드라인은 단서일 뿐이고, 사실 확인은 원문과 공시로 한다.

In [ ]:
nw = mk.news(tickers, per_ticker=3, refresh=REFRESH)
nw.insert(1, "기업", nw["ticker"].map(names))
esc = lambda x: str(x or "").replace("|", "/")
lines = ["| 기업 | 날짜 | 헤드라인 | 출처 |", "|---|---|---|---|"]
lines += [f"| {esc(r.기업)} | {esc(r.date)} | [{esc(r.title)}]({r.url}) | {esc(r.publisher)} |" if r.url
          else f"| {esc(r.기업)} | {esc(r.date)} | {esc(r.title)} | {esc(r.publisher)} |" for r in nw.itertuples()]
display(Markdown(chr(10).join(lines)))

## 5. 읽고 나서 적을 것

1. **시장은 이 밸류체인의 어디에 돈을 걸고 있나?** (어느 역할이 지수를 이기고, 어디가 지고 있나)
2. **내 관심 종목은 같은 역할 안에서 비싼가, 싼가? 그 차이를 01의 재무 숫자로 설명할 수 있나?**
3. **주가가 크게 움직인 종목은 무슨 일이 있었나?** (헤드라인은 단서, 확인은 공시로)

> 여기에 직접 작성하세요.